In [50]:
import numpy as np
import plotly.graph_objects as go
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from datetime import datetime, timezone
from sgp4.api import Satrec
import requests

print("Imports loaded!")

Imports loaded!


In [51]:
# Fetch and parse TLE
url = "https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE"
response = requests.get(url)
lines = response.text.strip().split('\n')

name  = lines[0].strip()
line1 = lines[1].strip()
line2 = lines[2].strip()

print(f"TLE fetched for: {name}")

# Parse
satellite = Satrec.twoline2rv(line1, line2)

# Epoch + age
epoch = Time(satellite.jdsatepoch + satellite.jdsatepochF, format='jd', scale='utc')
now   = Time(datetime.now(timezone.utc))
tle_age = (now - epoch).to(u.hour)

print(f"   Epoch:        {epoch.iso}")
print(f"   Current time: {now.iso}")
print(f"   TLE age:      {tle_age:.2f}")

if tle_age.value > 48:
    print(" WARNING: TLE is over 48 hours old — accuracy may be degraded")
elif tle_age.value > 24:
    print("TLE is over 24 hours old — consider refreshing")
else:
    print("TLE is fresh")

TLE fetched for: ISS (ZARYA)
   Epoch:        2026-04-14 20:15:54.269
   Current time: 2026-04-15 05:05:04.807
   TLE age:      8.82 h
TLE is fresh


In [52]:
# Ground Station — Cal Poly San Luis Obispo
# Birthplace of the CubeSat standard

gs_name = "Cal Poly SLO"
gs_lat  = 35.30   # degrees North
gs_lon  = -120.66 # degrees East
gs_alt  = 0.097   # km above sea level

print(f"   Ground station defined!")
print(f"   Name:      {gs_name}")
print(f"   Latitude:  {gs_lat}°N")
print(f"   Longitude: {gs_lon}°E")
print(f"   Altitude:  {gs_alt} km")

   Ground station defined!
   Name:      Cal Poly SLO
   Latitude:  35.3°N
   Longitude: -120.66°E
   Altitude:  0.097 km


In [53]:
# Current ISS position
e, r_teme, v_teme = satellite.sgp4(now.jd1, now.jd2)

r_teme_coord = CartesianRepresentation(r_teme * u.km)
teme = TEME(r_teme_coord, obstime=now)
itrs = teme.transform_to(ITRS(obstime=now))
loc = EarthLocation(*itrs.cartesian.xyz)

iss_lat_now = loc.lat.deg
iss_lon_now = loc.lon.deg
iss_alt_now = loc.height.to(u.km).value

print(f"  Current ISS position as of {now.iso} UTC")
print(f"   Latitude:  {iss_lat_now:.2f}°")
print(f"   Longitude: {iss_lon_now:.2f}°")
print(f"   Altitude:  {iss_alt_now:.2f} km")

  Current ISS position as of 2026-04-15 05:05:04.807 UTC
   Latitude:  -47.81°
   Longitude: -147.58°
   Altitude:  434.14 km


In [54]:
fig_gs = go.Figure()

# ── Ground track from now ─────────────────────────────────────────
def break_antimeridian(lons):
    lons_plot = lons.copy().astype(float)
    for i in range(1, len(lons_plot)):
        if abs(lons_plot[i] - lons_plot[i-1]) > 180:
            lons_plot[i-1] = None
    return lons_plot

# Propagate one orbit from now
num_steps = 500
mean_motion = satellite.no
period_s = (2 * np.pi / mean_motion) * 60
time_deltas = np.linspace(0, period_s, num_steps)

lats_now, lons_now = [], []
for dt in time_deltas:
    t = now + dt * u.s
    e, r_teme, v_teme = satellite.sgp4(t.jd1, t.jd2)
    r_teme_coord = CartesianRepresentation(r_teme * u.km)
    teme = TEME(r_teme_coord, obstime=t)
    itrs = teme.transform_to(ITRS(obstime=t))
    loc = EarthLocation(*itrs.cartesian.xyz)
    lats_now.append(loc.lat.deg)
    lons_now.append(loc.lon.deg)

lats_now = np.array(lats_now)
lons_now = np.array(lons_now)

# ── Ground track ──────────────────────────────────────────────────
fig_gs.add_trace(go.Scattergeo(
    lon=break_antimeridian(lons_now),
    lat=lats_now,
    mode='lines',
    line=dict(color='cyan', width=1.5),
    name='ISS Ground Track'
))

# ── ISS current position ──────────────────────────────────────────
fig_gs.add_trace(go.Scattergeo(
    lon=[iss_lon_now],
    lat=[iss_lat_now],
    mode='markers+text',
    marker=dict(size=8, color='red'),
    text=['ISS'],
    textfont=dict(color='white', size=11),
    name='ISS Now'
))

# ── Ground station ────────────────────────────────────────────────
fig_gs.add_trace(go.Scattergeo(
    lon=[gs_lon],
    lat=[gs_lat],
    mode='markers+text',
    marker=dict(size=10, color='orange', symbol='triangle-up'),
    text=[gs_name],
    textfont=dict(color='white', size=11),
    name=gs_name
))

fig_gs.update_layout(
    title=f'🛰️ ISS Ground Track + {gs_name}',
    geo=dict(
        showland=True, landcolor='darkgreen',
        showocean=True, oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='gray',
        showframe=False,
        bgcolor='black',
        projection_type='natural earth'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig_gs.show()
print(" Ground station plotted!")

 Ground station plotted!


In [55]:
# Visibility circle around Cal Poly SLO
R_earth = 6371.0  # km
elev_min = 10      # degrees minimum elevation (0 = horizon)

# Angular radius of visibility circle
elev_min_rad = np.radians(elev_min)
rho = np.arccos(R_earth / (R_earth + iss_alt_now)) - elev_min_rad

print(f"Minimum elevation angle: {elev_min}°")
print(f"Visibility circle radius: {np.degrees(rho):.1f}°")
print(f"Ground radius: {rho * R_earth:.0f} km")

# Walk 360° around ground station at angular distance rho
angles = np.linspace(0, 2 * np.pi, 361)
gs_lat_rad = np.radians(gs_lat)
gs_lon_rad = np.radians(gs_lon)

circle_lats = np.degrees(np.arcsin(
    np.sin(gs_lat_rad) * np.cos(rho) +
    np.cos(gs_lat_rad) * np.sin(rho) * np.cos(angles)
))

circle_lons = gs_lon + np.degrees(np.arctan2(
    np.sin(angles) * np.sin(rho) * np.cos(gs_lat_rad),
    np.cos(rho) - np.sin(gs_lat_rad) * np.sin(np.radians(circle_lats))
))

print(f" Visibility circle computed!")

Minimum elevation angle: 10°
Visibility circle radius: 10.6°
Ground radius: 1176 km
 Visibility circle computed!


In [56]:
fig_gs.add_trace(go.Scattergeo(
    lon=circle_lons,
    lat=circle_lats,
    mode='lines',
    line=dict(color='orange', width=1.5, dash='dot'),
    name=f'Visibility Circle ({gs_name})'
))

fig_gs.show()
print(" Visibility circle added!")

 Visibility circle added!


In [57]:
# Propagate ISS over 24 hours
num_steps = 2000  # more steps = more precision
duration_s = 24 * 3600  # 24 hours in seconds
time_deltas = np.linspace(0, duration_s, num_steps)

lats_24, lons_24, times_24 = [], [], []

for dt in time_deltas:
    t = now + dt * u.s
    e, r_teme, v_teme = satellite.sgp4(t.jd1, t.jd2)
    r_teme_coord = CartesianRepresentation(r_teme * u.km)
    teme = TEME(r_teme_coord, obstime=t)
    itrs = teme.transform_to(ITRS(obstime=t))
    loc = EarthLocation(*itrs.cartesian.xyz)
    lats_24.append(loc.lat.deg)
    lons_24.append(loc.lon.deg)
    times_24.append(t)

lats_24 = np.array(lats_24)
lons_24 = np.array(lons_24)

print(f" Propagated {num_steps} steps over 24 hours")

 Propagated 2000 steps over 24 hours


In [58]:
fig_24 = go.Figure()

fig_24.add_trace(go.Scattergeo(
    lon=break_antimeridian(lons_24),
    lat=lats_24,
    mode='lines',
    line=dict(color='cyan', width=1),
    name='24hr Ground Track'
))

fig_24.add_trace(go.Scattergeo(
    lon=[iss_lon_now],
    lat=[iss_lat_now],
    mode='markers+text',
    marker=dict(size=8, color='red'),
    text=['ISS'],
    textfont=dict(color='white', size=11),
    name='ISS Now'
))

fig_24.add_trace(go.Scattergeo(
    lon=[gs_lon],
    lat=[gs_lat],
    mode='markers+text',
    marker=dict(size=10, color='orange', symbol='triangle-up'),
    text=[gs_name],
    textfont=dict(color='white', size=11),
    name=gs_name
))

fig_24.add_trace(go.Scattergeo(
    lon=circle_lons,
    lat=circle_lats,
    mode='lines',
    line=dict(color='orange', width=1.5, dash='dot'),
    name='Visibility Circle'
))

fig_24.update_layout(
    title=f'🛰️ ISS 24hr Ground Track + {gs_name}',
    geo=dict(
        showland=True, landcolor='darkgreen',
        showocean=True, oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='gray',
        showframe=False, bgcolor='black',
        projection_type='natural earth'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig_24.show()
print(" 24hr ground track plotted!")

 24hr ground track plotted!


In [59]:
# Split 24hr track into individual orbits
period_steps = int(num_steps / (duration_s / period_s))  # steps per orbit
num_orbits = int(num_steps / period_steps)

orbits = []
for i in range(num_orbits):
    start = i * period_steps
    end = min((i + 1) * period_steps, num_steps)
    orbits.append({
        'lats': lats_24[start:end],
        'lons': lons_24[start:end],
        'times': times_24[start:end],
        'number': i + 1
    })

print(f" Split into {len(orbits)} orbits")
print(f"   Steps per orbit: {period_steps}")

 Split into 15 orbits
   Steps per orbit: 129


In [60]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors



# Visible light spectrum: red → orange → yellow → green → blue → indigo → violet
spectrum_colors = [
    '#FF0000',  # Red        - Orbit 1
    '#FF4500',  # Red-Orange - Orbit 2
    '#FF7000',  # Orange     - Orbit 3
    '#FFB000',  # Amber      - Orbit 4
    '#FFD700',  # Yellow     - Orbit 5
    '#ADFF2F',  # Yellow-Green - Orbit 6
    '#00FF00',  # Green      - Orbit 7
    '#00FF80',  # Green-Cyan - Orbit 8
    '#00FFFF',  # Cyan       - Orbit 9
    '#0080FF',  # Light Blue - Orbit 10
    '#0000FF',  # Blue       - Orbit 11
    '#4B0082',  # Indigo     - Orbit 12
    '#6A0DAD',  # Purple     - Orbit 13
    '#8B00FF',  # Violet     - Orbit 14
    '#9400D3',  # Dark Violet - Orbit 15
]

print(" Spectrum colors defined!")
for i, color in enumerate(spectrum_colors):
    print(f"   Orbit {i+1}: {color}")

fig_orbits = go.Figure()

for i, orbit in enumerate(orbits):
    fig_orbits.add_trace(go.Scattergeo(
        lon=break_antimeridian(orbit['lons']),
        lat=orbit['lats'],
        mode='lines',
        line=dict(color=spectrum_colors[i], width=1.5),
        name=f'Orbit {orbit["number"]}',
        visible=True if i == 0 else 'legendonly'
    ))

# ── Ground station ────────────────────────────────────────────────
fig_orbits.add_trace(go.Scattergeo(
    lon=[gs_lon],
    lat=[gs_lat],
    mode='markers+text',
    marker=dict(size=10, color='orange', symbol='triangle-up'),
    text=[gs_name],
    textfont=dict(color='white', size=11),
    name=gs_name,
    visible=True
))

# ── Visibility circle ─────────────────────────────────────────────
fig_orbits.add_trace(go.Scattergeo(
    lon=circle_lons,
    lat=circle_lats,
    mode='lines',
    line=dict(color='orange', width=1.5, dash='dot'),
    name='Visibility Circle',
    visible=True
))

# ── ISS current position ──────────────────────────────────────────
fig_orbits.add_trace(go.Scattergeo(
    lon=[iss_lon_now],
    lat=[iss_lat_now],
    mode='markers+text',
    marker=dict(size=8, color='red'),
    text=['ISS Now'],
    textfont=dict(color='white', size=11),
    name='ISS Now',
    visible=True
))

# ── Dropdown buttons ──────────────────────────────────────────────
buttons = [dict(
    label='All Orbits',
    method='update',
    args=[{'visible': [True] * len(orbits) + [True, True, True]}]
)]

for i in range(len(orbits)):
    visibility = [False] * len(orbits) + [True, True, True]
    visibility[i] = True
    buttons.append(dict(
        label=f'Orbit {i+1}',
        method='update',
        args=[{'visible': visibility}]
    ))

fig_orbits.update_layout(
    title=f'🛰️ ISS 24hr Ground Track + {gs_name}',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.1,
        y=1.15,
        bgcolor='black',
        font=dict(color='white')
    )],
    geo=dict(
        showland=True, landcolor='darkgreen',
        showocean=True, oceancolor='midnightblue',
        showcoastlines=True, coastlinecolor='white',
        showcountries=True, countrycolor='gray',
        showframe=False,
        bgcolor='black',
        projection_type='natural earth'
    ),
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)

fig_orbits.show()
print(" Orbit dropdown plotted!")

 Spectrum colors defined!
   Orbit 1: #FF0000
   Orbit 2: #FF4500
   Orbit 3: #FF7000
   Orbit 4: #FFB000
   Orbit 5: #FFD700
   Orbit 6: #ADFF2F
   Orbit 7: #00FF00
   Orbit 8: #00FF80
   Orbit 9: #00FFFF
   Orbit 10: #0080FF
   Orbit 11: #0000FF
   Orbit 12: #4B0082
   Orbit 13: #6A0DAD
   Orbit 14: #8B00FF
   Orbit 15: #9400D3


 Orbit dropdown plotted!


In [61]:
# Convert ground station lat/lon to radians
# We do this once here instead of inside the function 
# so we're not repeating the conversion 2000 times in the loop
gs_lat_rad = np.radians(gs_lat)
gs_lon_rad = np.radians(gs_lon)

def in_visibility_circle(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    cos_sep = (np.sin(gs_lat_rad) * np.sin(lat_rad) +
               np.cos(gs_lat_rad) * np.cos(lat_rad) * np.cos(lon_rad - gs_lon_rad))
    
    sep = np.arccos(np.clip(cos_sep, -1, 1))
    return sep < rho
result = in_visibility_circle(iss_lat_now, iss_lon_now)
print(f"Function working!")
print(f"ISS currently within visibility circle: {result}")




Function working!
ISS currently within visibility circle: False


In [62]:
'''
# Step 2 - check every propagated point against visibility circle
inside = np.array([in_visibility_circle(lats_24[i], lons_24[i]) 
                   for i in range(len(lons_24))])

print(f"   Checked {len(inside)} points")
print(f"   Points inside visibility circle: {inside.sum()}")
print(f"   Points outside visibility circle: {(~inside).sum()}") '''

'\n# Step 2 - check every propagated point against visibility circle\ninside = np.array([in_visibility_circle(lats_24[i], lons_24[i]) \n                   for i in range(len(lons_24))])\n\nprint(f"   Checked {len(inside)} points")\nprint(f"   Points inside visibility circle: {inside.sum()}")\nprint(f"   Points outside visibility circle: {(~inside).sum()}") '

In [63]:
'''
# Step 3 - group consecutive True points into passes
passes = []
in_pass = False

for i in range(len(inside)):
    if inside[i] and not in_pass:
        pass_start = i
        in_pass = True
    elif not inside[i] and in_pass:
        passes.append((pass_start, i - 1))
        in_pass = False

# Handle case where pass extends to end of window
if in_pass:
    passes.append((pass_start, len(inside) - 1))

time_between_points = duration_s / num_steps  # seconds between each propagated point

print(f" Found {len(passes)} passes over {gs_name} in next 24 hours")
for i, (start, end) in enumerate(passes):
    orbit_num = start // period_steps + 1
    duration_pass = (end - start) * time_between_points
    t_start = times_24[start]
    t_end   = times_24[end]
    print(f"   Pass {i+1}: Orbit {orbit_num} | Start: {t_start.iso} UTC | End: {t_end.iso} UTC | Duration: {duration_pass/60:.1f} min")
    '''

'\n# Step 3 - group consecutive True points into passes\npasses = []\nin_pass = False\n\nfor i in range(len(inside)):\n    if inside[i] and not in_pass:\n        pass_start = i\n        in_pass = True\n    elif not inside[i] and in_pass:\n        passes.append((pass_start, i - 1))\n        in_pass = False\n\n# Handle case where pass extends to end of window\nif in_pass:\n    passes.append((pass_start, len(inside) - 1))\n\ntime_between_points = duration_s / num_steps  # seconds between each propagated point\n\nprint(f" Found {len(passes)} passes over {gs_name} in next 24 hours")\nfor i, (start, end) in enumerate(passes):\n    orbit_num = start // period_steps + 1\n    duration_pass = (end - start) * time_between_points\n    t_start = times_24[start]\n    t_end   = times_24[end]\n    print(f"   Pass {i+1}: Orbit {orbit_num} | Start: {t_start.iso} UTC | End: {t_end.iso} UTC | Duration: {duration_pass/60:.1f} min")\n    '

In [64]:
alts_24 = []
for t in times_24:
    e, r_teme, v_teme = satellite.sgp4(t.jd1, t.jd2)
    r_teme_coord = CartesianRepresentation(r_teme * u.km)
    teme = TEME(r_teme_coord, obstime=t)
    itrs = teme.transform_to(ITRS(obstime=t))
    loc = EarthLocation(*itrs.cartesian.xyz)
    alts_24.append(loc.height.to(u.km).value)

print(f"alts_24 ready — {len(alts_24)} points, range {min(alts_24):.1f}–{max(alts_24):.1f} km")

alts_24 ready — 2000 points, range 417.1–434.9 km


In [65]:
def elevation_angle(lat, lon, alt_km):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    
    cos_sep = (np.sin(gs_lat_rad) * np.sin(lat_rad) +
               np.cos(gs_lat_rad) * np.cos(lat_rad) * np.cos(lon_rad - gs_lon_rad))
    sep = np.arccos(np.clip(cos_sep, -1, 1))
    
    R_e = 6371.0  # km
    r_sat = R_e + alt_km
    
    el = np.arctan2(np.cos(sep) - R_e / r_sat, np.sin(sep))
    return np.degrees(el)

In [66]:
#d2d2d
min_elev = 10.0  # degrees
elevations = np.array([elevation_angle(lats_24[i], lons_24[i], alts_24[i])
                        for i in range(len(lats_24))])
inside = elevations > min_elev
print(f"Elevations range: {elevations.min():.1f}° to {elevations.max():.1f}°")
print(f"Points above {min_elev}°: {inside.sum()} / {len(inside)}")

Elevations range: -89.4° to 69.1°
Points above 10.0°: 30 / 2000


In [67]:
# Step 3 - group consecutive True points into passes
passes = []
in_pass = False

for i in range(len(inside)):
    if inside[i] and not in_pass:
        pass_start = i
        in_pass = True
    elif not inside[i] and in_pass:
        passes.append((pass_start, i - 1))
        in_pass = False

# Handle case where pass extends to end of window
if in_pass:
    passes.append((pass_start, len(inside) - 1))

time_between_points = duration_s / num_steps  # seconds between each propagated point

print(f" Found {len(passes)} passes over {gs_name} in next 24 hours")
for i, (start, end) in enumerate(passes):
    orbit_num = start // period_steps + 1
    duration_pass = (end - start) * time_between_points
    t_start = times_24[start]
    t_end   = times_24[end]
    print(f"   Pass {i+1}: Orbit {orbit_num} | Start: {t_start.iso} UTC | End: {t_end.iso} UTC | Duration: {duration_pass/60:.1f} min")

 Found 4 passes over Cal Poly SLO in next 24 hours
   Pass 1: Orbit 6 | Start: 2026-04-15 13:27:10.269 UTC | End: 2026-04-15 13:33:39.264 UTC | Duration: 6.5 min
   Pass 2: Orbit 7 | Start: 2026-04-15 15:06:34.852 UTC | End: 2026-04-15 15:08:44.516 UTC | Duration: 2.2 min
   Pass 3: Orbit 10 | Start: 2026-04-15 19:59:02.826 UTC | End: 2026-04-15 20:04:48.599 UTC | Duration: 5.8 min
   Pass 4: Orbit 11 | Start: 2026-04-15 21:36:17.743 UTC | End: 2026-04-15 21:40:37.073 UTC | Duration: 4.3 min


In [74]:
for i, (start, end) in enumerate(passes):
    print(f"Pass {i+1}: {end - start + 1} timesteps")

for i, (start, end) in enumerate(passes):
    print(f"\nPass {i+1}:")
    print(f"{'Time (UTC)':<35} {'Elevation (°)'}")
    print("-" * 50)
    for j in range(start, end+1):
        print(f"{times_24[j].iso:<35} {elevations[j]:.2f}°")

Pass 1: 10 timesteps
Pass 2: 4 timesteps
Pass 3: 9 timesteps
Pass 4: 7 timesteps

Pass 1:
Time (UTC)                          Elevation (°)
--------------------------------------------------
2026-04-15 13:27:10.269             11.34°
2026-04-15 13:27:53.491             17.36°
2026-04-15 13:28:36.713             26.48°
2026-04-15 13:29:19.934             42.19°
2026-04-15 13:30:03.156             69.12°
2026-04-15 13:30:46.377             61.56°
2026-04-15 13:31:29.599             37.36°
2026-04-15 13:32:12.821             23.90°
2026-04-15 13:32:56.042             15.82°
2026-04-15 13:33:39.264             10.32°

Pass 2:
Time (UTC)                          Elevation (°)
--------------------------------------------------
2026-04-15 15:06:34.852             10.84°
2026-04-15 15:07:18.073             11.89°
2026-04-15 15:08:01.295             11.83°
2026-04-15 15:08:44.516             10.70°

Pass 3:
Time (UTC)                          Elevation (°)
--------------------------------------

In [75]:
fig_elev = go.Figure()

for i, (start, end) in enumerate(passes):
    t_slice = times_24[start:end+1]
    el_slice = elevations[start:end+1]
    peak_el = el_slice.max()
    peak_idx = el_slice.argmax()
    peak_time = t_slice[peak_idx]
    
    # Convert to datetime for plotting
    t_plot = [t.to_datetime(timezone.utc) for t in t_slice]
    
    fig_elev.add_trace(go.Scatter(
        x=t_plot,
        y=el_slice,
        mode='lines',
        name=f'Pass {i+1} | Peak {peak_el:.1f}°',
        line=dict(width=2)
    ))
    
    # Annotate peak
    fig_elev.add_annotation(
        x=peak_time.to_datetime(timezone.utc),
        y=peak_el,
        text=f'{peak_el:.1f}°',
        showarrow=True,
        arrowhead=2,
        font=dict(color='white', size=10)
    )

# Min elevation reference line
fig_elev.add_hline(
    y=min_elev,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Min elevation {min_elev}°',
    annotation_font_color='red'
)

fig_elev.update_layout(
    title=f'🛰️ ISS Elevation Arcs — {gs_name}',
    xaxis_title='Time (UTC)',
    yaxis_title='Elevation (°)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray', range=[0, 90]),
    legend=dict(bgcolor='black')
)

fig_elev.show()
print(f" Elevation arcs plotted for {len(passes)} passes")
for i, (start, end) in enumerate(passes):
    el_slice = elevations[start:end+1]
    print(f"   Pass {i+1}: Peak {el_slice.max():.1f}° at {times_24[start + el_slice.argmax()].iso} UTC")


 Elevation arcs plotted for 4 passes
   Pass 1: Peak 69.1° at 2026-04-15 13:30:03.156 UTC
   Pass 2: Peak 11.9° at 2026-04-15 15:07:18.073 UTC
   Pass 3: Peak 31.6° at 2026-04-15 20:01:55.712 UTC
   Pass 4: Peak 21.6° at 2026-04-15 21:38:27.408 UTC
